# Silver Layer: Customer Info CDC (Free Edition)

**Pattern**: CDC via `foreachBatch` Merge  
**Trigger**: AvailableNow (Incremental Batch)

In [0]:
from pyspark.sql.functions import col, trim, when, upper
from delta.tables import DeltaTable

# Paths and Tables
bronze_table = "workspace.bronze.crm_cust_info_cdc"
silver_table = "workspace.silver.crm_customers_cdc"
checkpoint_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/silver_crm_cust_info"

def upsert_to_silver(batch_df, batch_id):
    transformed_df = (
        batch_df.select(
            col("cst_id").alias("customer_id"),
            col("cst_key").alias("customer_number"),
            trim(col("cst_firstname")).alias("first_name"),
            trim(col("cst_lastname")).alias("last_name"),
            when(upper(col("cst_marital_status")) == "S", "Single")
                .when(upper(col("cst_marital_status")) == "M", "Married")
                .otherwise("n/a").alias("marital_status"),
            when(upper(col("cst_gndr")) == "F", "Female")
                .when(upper(col("cst_gndr")) == "M", "Male")
                .otherwise("n/a").alias("gender"),
            col("cst_create_date").alias("created_date"),
            col("ingestion_timestamp")
        )
        .filter(col("customer_id").isNotNull())
    )

    if not spark.catalog.tableExists(silver_table):
        transformed_df.write.format("delta").saveAsTable(silver_table)
        return

    target_table = DeltaTable.forName(spark, silver_table)
    (target_table.alias("t")
        .merge(transformed_df.alias("s"), "t.customer_id = s.customer_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

print(f"Starting incremental update for {silver_table}...")
query = (
    spark.readStream
    .table(bronze_table)
    .writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print(f"✓ Silver update complete. Total records in {silver_table}: {spark.table(silver_table).count():,}")

In [0]:
display(spark.table("workspace.silver.crm_customers_cdc").count())
